# Lab — Uninformed and Informed Search: Maze and Road Map
**AI · Level 6 · Class 2 — Search Strategies**

This single notebook builds ONE generic search function and reuses it on
two different problems: a maze  and a road map.
Both problems plug into the same `search()` / `search_tree()` engine —
**BFS**, **DFS**, **UCS** and the informed strategies **Greedy best-first**
and **A\*** — so you can compare uninformed and informed search side by
side on two very different state spaces.

Pick which problem to solve in the "Problem definitions" section further
down (`PROBLEM = "maze"` or `PROBLEM = "roadmap"`, plus a maze/road map
number). Run this notebook top to bottom in Google Colab. No installation
needed, only the Python standard library.

In [6]:
import time
import heapq
import itertools
import math
from collections import deque

## 1. Problem formulation

Every search problem is defined by 5 elements: the initial state, the
actions available from a state, the transition model (`result`), the goal
test, and the step cost. We express that as a small base class.

In [7]:
class Problem:
    """Generic formulation of a search problem (the 5 elements)."""

    def __init__(self, initial_state):
        self.initial_state = initial_state

    def actions(self, state):
        raise NotImplementedError

    def result(self, state, action):
        raise NotImplementedError

    def is_goal(self, state):
        raise NotImplementedError

    def step_cost(self, state, action, next_state):
        raise NotImplementedError

## 2. Node and path reconstruction

A `Node` wraps a state with the information we need to rebuild the path
once we reach the goal: who its parent is, which action led to it, the
accumulated cost g(n), and its depth.

In [8]:
class Node:
    __slots__ = ("state", "parent", "action", "path_cost", "depth")

    def __init__(self, state, parent=None, action=None, path_cost=0, depth=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost
        self.depth = depth

    def __repr__(self):
        return f"Node({self.state!r}, g={self.path_cost}, depth={self.depth})"


def reconstruct_path(node):
    """Walk the parent pointers back to the root and reverse the result."""
    states, actions = [], []
    while node is not None:
        states.append(node.state)
        if node.action is not None:
            actions.append(node.action)
        node = node.parent
    states.reverse()
    actions.reverse()
    return states, actions


def on_path(node, state):
    """Whether `state` is on the root-to-`node` path (including `node`
    itself), walking the parent pointers without building any extra
    structure. Used by search()/search_tree() when
    repeated="Remember_path_checking"."""
    while node is not None:
        if node.state == state:
            return True
        node = node.parent
    return False


## 3. The generic search algorithm

BFS, DFS, UCS, Greedy best-first and A\* are **the same algorithm**. The
only thing that changes is how the frontier is managed and, for the last
two, that a `heuristic` is required:

| Strategy | Frontier type      | Priority / Extract        | Add            |
|----------|---------------------|----------------------------|----------------|
| BFS      | FIFO queue          | oldest (`popleft`)         | append at back |
| DFS      | LIFO stack          | newest (`pop`)              | append at end  |
| UCS      | priority queue      | lowest `g(n)`               | `heappush`     |
| Greedy   | priority queue      | lowest `h(n)`               | `heappush`     |
| A\*      | priority queue      | lowest `g(n) + h(n)`        | `heappush`     |

The setup below (given) prepares three small functions — `add`, `extract`,
`is_empty` — that already know how to do the right thing for the chosen
`strategy`. The job in the main loop is just to **call them**: those 3
calls are exactly the 3 lines that make BFS, DFS and UCS behave
differently, even though they look identical. Greedy and A\* reuse the
very same idea with a different priority.

The goal is tested when a node is **extracted**, not when it is generated —
this matters most for UCS and A\* (see the class slides): a node already in
the frontier is not accepted until it comes out with the lowest cost. This
is exactly what makes UCS able to **replace** an expensive route to the
goal in the frontier with a cheaper one discovered later (see the road map
problem below, where roads have different costs).

`search()` and `search_tree()` take a `repeated` argument that controls repeated-state handling: `"Remember_all_visited"` (the default) keeps the classic `visited` set — discarded on extraction, pruned on generation; `"Remember_path_checking"` drops the `visited` set and instead prunes a child only when its state is already on the current root-to-node path, checked with `on_path(node, state)` (walks `node.parent` without building any extra structure); `"No_control"` applies no repeated-state control at all. An unknown `repeated` value raises `ValueError`.

Both functions also report `max_memory`, the peak — right after each expansion adds its children to the frontier — of `len(frontier) + len(visited)` (mode `"Remember_all_visited"`) or `len(frontier) + (node.depth + 1)`, the current path length (modes `"Remember_path_checking"` and `"No_control"`).

In [9]:
def search(problem, strategy, repeated="Remember_all_visited", max_expansions=10_000, heuristic=None):
    """Run BFS, DFS, UCS, Greedy best-first or A* on `problem` and return a metrics dictionary.

    max_expansions stops the search safely and reports limit_reached=True,
    so a runaway DFS never freezes Colab.
    heuristic is required for "greedy" and "astar": either a callable
    h(state) -> float, or the name of an entry in HEURISTICS (e.g.
    "manhattan"), which is resolved against GOAL the moment search() runs.
    repeated controls repeated-state handling: "Remember_all_visited" (the
    default) keeps the classic visited set (discarded on extraction, pruned
    on generation); "Remember_path_checking" drops the visited set and
    instead prunes a child only when its state is already on the current
    root-to-node path (via on_path(node, state)); "No_control" applies no
    repeated-state control at all. Any other value raises ValueError.
    """
    if repeated not in ("No_control", "Remember_path_checking", "Remember_all_visited"):
        raise ValueError(
            f"Unknown repeated mode: {repeated!r}. Use 'No_control', "
            f"'Remember_path_checking' or 'Remember_all_visited'."
        )

    start_time = time.perf_counter()
    root = Node(problem.initial_state)
    counter = itertools.count()  # heapq tie-breaker: Node objects are not comparable

    def _resolve_heuristic(h):
        # Resolved at call time (not at definition time) so that a
        # heuristic name can be looked up in HEURISTICS / GOAL even
        # though search() is defined earlier in the notebook than they are.
        if h is None:
            raise ValueError(
                f"strategy {strategy!r} needs a heuristic: pass a callable "
                f"h(state) or a name in HEURISTICS, e.g. heuristic=\"manhattan\""
            )
        if callable(h):
            return h
        if isinstance(h, str):
            if h not in HEURISTICS:
                raise ValueError(f"Unknown heuristic name: {h!r}. Available: {sorted(HEURISTICS)}")
            h_func = HEURISTICS[h]
            return lambda state: h_func(state, GOAL)
        raise TypeError("heuristic must be a callable h(state) or a string name")

    if strategy == "bfs":
        frontier = deque([root])
        add = frontier.append
        extract = frontier.popleft
    elif strategy == "dfs":
        frontier = [root]
        add = frontier.append
        extract = frontier.pop
    elif strategy == "ucs":
        frontier = []
        heapq.heappush(frontier, (root.path_cost, next(counter), root))

        def add(node):
            heapq.heappush(frontier, (node.path_cost, next(counter), node))

        def extract():
            _, _, node = heapq.heappop(frontier)
            return node
    elif strategy in ("greedy", "astar"):
        h = _resolve_heuristic(heuristic)

        def priority(node):
            return h(node.state) if strategy == "greedy" else node.path_cost + h(node.state)

        frontier = []
        heapq.heappush(frontier, (priority(root), next(counter), root))

        def add(node):
            heapq.heappush(frontier, (priority(node), next(counter), node))

        def extract():
            _, _, node = heapq.heappop(frontier)
            return node
    else:
        raise ValueError(f"Unknown strategy: {strategy}")

    def is_empty():
        return len(frontier) == 0

    visited = set()
    nodes_expanded = 0
    max_frontier_size = 1
    max_memory = 1

    def finish(found, node=None, limit_reached=False):
        elapsed_ms = (time.perf_counter() - start_time) * 1000
        if found:
            states, actions = reconstruct_path(node)
            return {"found": True, "path": states, "steps": len(actions),
                    "path_cost": node.path_cost, "nodes_expanded": nodes_expanded,
                    "max_frontier_size": max_frontier_size, "max_memory": max_memory,
                    "time_ms": elapsed_ms, "limit_reached": False}
        return {"found": False, "path": [], "steps": 0, "path_cost": None,
                "nodes_expanded": nodes_expanded, "max_frontier_size": max_frontier_size,
                "max_memory": max_memory, "time_ms": elapsed_ms, "limit_reached": limit_reached}

    while True:
        if is_empty():
            return finish(False)

        if nodes_expanded >= max_expansions:
            return finish(False, limit_reached=True)

        node = extract()

        if problem.is_goal(node.state):
            return finish(True, node)

        if repeated == "Remember_all_visited":
            if node.state in visited:
                continue
            visited.add(node.state)

        nodes_expanded += 1

        child_actions = problem.actions(node.state)
        if strategy == "dfs":
            # DFS pops the LAST child pushed (LIFO). Reversing here makes it
            # explore actions in the order actions() lists them (e.g. "Up"
            # before "Right"), matching what you read in the code.
            child_actions = list(reversed(child_actions))

        for action in child_actions:
            next_state = problem.result(node.state, action)
            step_cost = problem.step_cost(node.state, action, next_state)
            child = Node(next_state, node, action, node.path_cost + step_cost, node.depth + 1)

            if repeated == "Remember_all_visited" and child.state in visited:
                continue
            if repeated == "Remember_path_checking" and on_path(node, next_state):
                continue

            add(child)

        max_frontier_size = max(max_frontier_size, len(frontier))
        if repeated == "Remember_all_visited":
            max_memory = max(max_memory, len(frontier) + len(visited))
        else:
            max_memory = max(max_memory, len(frontier) + (node.depth + 1))


### Visualizing the search tree

`search_tree()` runs **exactly the same rules** as `search()` (same frontier
per strategy, goal test on extraction, same repeated-state handling (`repeated`), same
heuristic resolution) but, instead of returning only the final path, it
records every node generated and prints the **search tree** that the
strategy has built. It supports all five strategies — **BFS**, **DFS**,
**UCS**, **Greedy best-first** and **A\*** — the last two need `heuristic`
just like `search()`:

- `[#k]` — the node was the k-th one **expanded** (order in which the strategy explores).
- `GOAL` — node extracted and accepted as the solution.
- `(in frontier)` — generated but never extracted: the search stopped first.
- `(discarded: already visited)` — extracted, but its state had already been expanded.
- `(pruned: already visited)` — not even added to the frontier, its state was already expanded (mode `"Remember_all_visited"`).
- `(pruned: already on path)` — not even added to the frontier, its state was already on the current root-to-node path (mode `"Remember_path_checking"`).
- `*` marks the nodes on the solution path.

For the informed strategies (greedy/astar) each node also shows its
heuristic value `h`, next to the accumulated cost `g`, so you can see the
priority the strategy used to order the frontier.

In [10]:
def search_tree(problem, strategy, repeated="Remember_all_visited", max_expansions=10_000, heuristic=None):
    """Same rules as search(), but it builds and prints the search tree.

    Supports the same five strategies as search(): "bfs", "dfs", "ucs",
    "greedy" and "astar". heuristic is required for "greedy" and "astar",
    resolved exactly like in search() (a callable h(state) or a name in
    HEURISTICS, looked up against GOAL at call time). repeated works exactly
    like in search() too: "Remember_all_visited" (the default), "Remember_path_checking"
    (prunes on on_path(node, state), printed as "(pruned: already on
    path)"), "No_control" (no control at all). Any other value raises
    ValueError.

    Returns the same metrics dictionary as search(), including max_memory."""
    if repeated not in ("No_control", "Remember_path_checking", "Remember_all_visited"):
        raise ValueError(
            f"Unknown repeated mode: {repeated!r}. Use 'No_control', "
            f"'Remember_path_checking' or 'Remember_all_visited'."
        )

    start_time = time.perf_counter()
    root = Node(problem.initial_state)
    counter = itertools.count()
    children = {id(root): []}     # id(node) -> list of (child_node | pruned_state)
    status = {}                    # id(node) -> "#k", "GOAL" or "discarded"
    step_costs = {}                # id(node) -> cost of the edge from its parent

    def _resolve_heuristic(h):
        # Resolved at call time (not at definition time) so that a
        # heuristic name can be looked up in HEURISTICS / GOAL even
        # though search_tree() is defined earlier in the notebook than they are.
        if h is None:
            raise ValueError(
                f"strategy {strategy!r} needs a heuristic: pass a callable "
                f"h(state) or a name in HEURISTICS, e.g. heuristic=\"manhattan\""
            )
        if callable(h):
            return h
        if isinstance(h, str):
            if h not in HEURISTICS:
                raise ValueError(f"Unknown heuristic name: {h!r}. Available: {sorted(HEURISTICS)}")
            h_func = HEURISTICS[h]
            return lambda state: h_func(state, GOAL)
        raise TypeError("heuristic must be a callable h(state) or a string name")

    h = None
    if strategy == "bfs":
        frontier = deque([root])
        add, extract = frontier.append, frontier.popleft
    elif strategy == "dfs":
        frontier = [root]
        add, extract = frontier.append, frontier.pop
    elif strategy == "ucs":
        frontier = [(0, next(counter), root)]

        def add(node):
            heapq.heappush(frontier, (node.path_cost, next(counter), node))

        def extract():
            return heapq.heappop(frontier)[2]
    elif strategy in ("greedy", "astar"):
        h = _resolve_heuristic(heuristic)

        def priority(node):
            return h(node.state) if strategy == "greedy" else node.path_cost + h(node.state)

        frontier = [(priority(root), next(counter), root)]

        def add(node):
            heapq.heappush(frontier, (priority(node), next(counter), node))

        def extract():
            return heapq.heappop(frontier)[2]
    else:
        raise ValueError(f"Unknown strategy: {strategy}")

    visited = set()
    nodes_expanded = 0
    max_frontier_size = 1
    max_memory = 1
    goal_node = None
    limit_reached = False

    while frontier:
        if nodes_expanded >= max_expansions:
            limit_reached = True
            break
        node = extract()
        if problem.is_goal(node.state):
            status[id(node)] = "GOAL"
            goal_node = node
            break
        if repeated == "Remember_all_visited":
            if node.state in visited:
                status[id(node)] = "discarded"
                continue
            visited.add(node.state)
        nodes_expanded += 1
        status[id(node)] = f"#{nodes_expanded}"

        generated = []
        for action in problem.actions(node.state):   # natural order, for display
            next_state = problem.result(node.state, action)
            cost = problem.step_cost(node.state, action, next_state)
            if repeated == "Remember_all_visited" and next_state in visited:
                generated.append(("pruned", next_state, cost))
                continue
            if repeated == "Remember_path_checking" and on_path(node, next_state):
                generated.append(("pruned_path", next_state, cost))
                continue
            child = Node(next_state, node, action, node.path_cost + cost, node.depth + 1)
            children[id(child)] = []
            step_costs[id(child)] = cost
            generated.append(("node", child, cost))
        children[id(node)] = generated

        to_add = [c for kind, c, _ in generated if kind == "node"]
        if strategy == "dfs":
            to_add.reverse()   # so that the first neighbour is extracted first
        for child in to_add:
            add(child)
        max_frontier_size = max(max_frontier_size, len(frontier))
        if repeated == "Remember_all_visited":
            max_memory = max(max_memory, len(frontier) + len(visited))
        else:
            max_memory = max(max_memory, len(frontier) + (node.depth + 1))

    # ---- print the tree -------------------------------------------------
    path_node_ids = set()
    n = goal_node
    while n is not None:
        path_node_ids.add(id(n))
        n = n.parent

    def label(node):
        mark = "*" if id(node) in path_node_ids else " "
        s = status.get(id(node))
        if s == "GOAL":
            tag = "GOAL"
        elif s == "discarded":
            tag = "(discarded: already visited)"
        elif s is None:
            tag = "(in frontier)"
        else:
            tag = f"[{s}]"
        if h is not None:
            return f"{mark}{node.state} (g={node.path_cost}, h={h(node.state)}) {tag}"
        return f"{mark}{node.state} (g={node.path_cost}) {tag}"

    def draw(node, prefix):
        # Iterative (explicit stack) instead of recursive: with
        # repeated="No_control"/"Remember_path_checking" a chain can be far
        # deeper than Python's call-stack recursion limit, so the tree still
        # has to print correctly without a RecursionError. Each stack frame
        # is [node, prefix, items, next_index]; pushing a child frame mimics
        # "descend immediately", exactly like the equivalent recursive call.
        stack = [[node, prefix, children.get(id(node), []), 0]]
        while stack:
            frame = stack[-1]
            cur_node, cur_prefix, items, idx = frame
            if idx >= len(items):
                stack.pop()
                continue
            frame[3] += 1
            kind, item, cost = items[idx]
            last = idx == len(items) - 1
            branch = "\u2514\u2500\u2500 " if last else "\u251c\u2500\u2500 "
            if kind == "pruned":
                print(f"{cur_prefix}{branch}--{cost}--> {item} (pruned: already visited)")
            elif kind == "pruned_path":
                print(f"{cur_prefix}{branch}--{cost}--> {item} (pruned: already on path)")
            else:
                print(f"{cur_prefix}{branch}--{cost}--> {label(item)}")
                child_prefix = cur_prefix + ("    " if last else "\u2502   ")
                stack.append([item, child_prefix, children.get(id(item), []), 0])

    print(f"Search tree \u2014 {strategy.upper()}  ({problem.initial_state} -> goal)")
    print(label(root))
    draw(root, "")

    if goal_node is not None:
        states, actions = reconstruct_path(goal_node)
        print(f"\nSolution: {' -> '.join(str(s) for s in states)}  "
              f"({len(actions)} step(s), cost {goal_node.path_cost}, "
              f"{nodes_expanded} node(s) expanded)")
        return {"found": True, "path": states, "steps": len(actions),
                "path_cost": goal_node.path_cost, "nodes_expanded": nodes_expanded,
                "max_frontier_size": max_frontier_size, "max_memory": max_memory,
                "time_ms": (time.perf_counter() - start_time) * 1000, "limit_reached": False}
    print("\nNo solution found" + (" (expansion limit reached)" if limit_reached else ""))
    return {"found": False, "path": [], "steps": 0, "path_cost": None,
            "nodes_expanded": nodes_expanded, "max_frontier_size": max_frontier_size,
            "max_memory": max_memory,
            "time_ms": (time.perf_counter() - start_time) * 1000, "limit_reached": limit_reached}


In [11]:
class _ToyProblem(Problem):
    _GRAPH = {
        "START": [("A", 1)], "A": [("B", 5), ("C", 1)],
        "B": [("D", 1)], "C": [("D", 1)], "D": [("G", 1)], "G": [],
    }

    def __init__(self):
        super().__init__("START")

    def is_goal(self, state):
        return state == "G"

    def actions(self, state):
        return [city for city, _ in self._GRAPH[state]]

    def result(self, state, action):
        return action

    def step_cost(self, state, action, next_state):
        return dict(self._GRAPH[state])[action]


_toy = _ToyProblem()
_bfs = search(_toy, "bfs")
_ucs = search(_toy, "ucs")
_astar_zero = search(_toy, "astar", heuristic=lambda state: 0)
assert _bfs["found"] and _bfs["steps"] == 4 and _bfs["path_cost"] == 8, "BFS should find the 4-step, cost-8 path via B"
assert _ucs["found"] and _ucs["path_cost"] == 4 and _ucs["path"] == ["START", "A", "C", "D", "G"], "UCS should find the cheaper path via C"
assert _astar_zero["found"] and _astar_zero["path_cost"] == 4 and _astar_zero["path"] == ["START", "A", "C", "D", "G"], "A* with h=0 should behave like UCS on this toy graph"
print("\u2705 correct \u2014 your search() implementation passes the toy self-check")

✅ correct — your search() implementation passes the toy self-check


In [12]:
# Choose the problem and the specific instance to load. The numbers match
# the class slides: MAZE_TYPE is 0, 1, 2 or 3; ROADMAP_TYPE is 1 to 10.
# Re-run the notebook from this cell onward after changing any of them.
PROBLEM = "roadmap"        # "maze" or "roadmap"
MAZE_TYPE = 3            # 0, 1, 2 or 3
ROADMAP_TYPE = 5         # 1 to 10

MAZES = {
    # MAZE 0 — empty grid, no walls
    0: set(),
    # MAZE 1
    1: {
        (1, 4),
        (2, 2), (2, 4), (2, 6), (2, 7), (2, 8),
        (3, 2), (3, 6),
        (4, 2), (4, 3), (4, 4), (4, 6), (4, 8), (4, 9),
        (5, 8),
        (6, 1), (6, 2), (6, 4), (6, 5), (6, 6), (6, 8),
        (7, 4), (7, 6),
    },
    # MAZE 2
    2: {
        (1, 4),
        (2, 2), (2, 4), (2, 6), (2, 7), (2, 8),
        (3, 2),
        (4, 2), (4, 3), (4, 4), (4, 6), (4, 8), (4, 9),
        (5, 8),
        (6, 1), (6, 2), (6, 4), (6, 5), (6, 6), (6, 8),
        (7, 4), (7, 6),
    },
    # MAZE 3
    3: {
        (2, 4), (2, 5), (2, 9),
        (3, 2), (3, 4), (3, 5), (3, 8), (3, 9),
        (4, 1), (4, 3), (4, 4), (4, 7), (4, 8), (4, 9),
        (5, 1), (5, 4), (5, 6), (5, 7),
        (6, 1), (6, 4), (6, 7), (6, 9),
        (7, 2),
        (8, 3), (8, 8),
    },
}

ROADMAPS = {
    # ROADMAP 1 — same cost either way (direct road vs via Cuenca)
    1: {
        "roads": {
            ("Madrid", "Cuenca"): 150,
            ("Madrid", "Valencia"): 400,
            ("Madrid", "Albacete"): 220,
            ("Cuenca", "Valencia"): 250,
            ("Albacete", "Valencia"): 180,
        },
        "start": "Madrid",
        "goal": "Valencia",
    },
    # ROADMAP 2 — same roads and cost as ROADMAP 1, listed in a different order
    2: {
        "roads": {
            ("Madrid", "Valencia"): 400,
            ("Madrid", "Cuenca"): 150,
            ("Madrid", "Albacete"): 220,
            ("Cuenca", "Valencia"): 250,
            ("Albacete", "Valencia"): 180,
        },
        "start": "Madrid",
        "goal": "Valencia",
    },
    # ROADMAP 3 — 9 nodes expanded; can reordering neighbours reduce it?
    3: {
        "roads": {
            ("Madrid", "Toledo"): 100,
            ("Toledo", "Ciudad Real"): 120,
            ("Ciudad Real", "Cordoba"): 180,
            ("Cordoba", "Sevilla"): 140,
            ("Sevilla", "Huelva"): 90,
            ("Huelva", "Badajoz"): 130,
            ("Badajoz", "Caceres"): 80,
            ("Caceres", "Salamanca"): 70,
            ("Salamanca", "Leon"): 90,
            ("Madrid", "Ciudad Real"): 260,
            ("Toledo", "Cordoba"): 350,
            ("Sevilla", "Badajoz"): 260,
            ("Caceres", "Leon"): 220,
        },
        "start": "Madrid",
        "goal": "Leon",
    },
    # ROADMAP 4 — reduces from 9 to 8 nodes expanded
    4: {
        "roads": {
            ("Madrid", "Toledo"): 100,
            ("Toledo", "Ciudad Real"): 120,
            ("Ciudad Real", "Cordoba"): 180,
            ("Cordoba", "Sevilla"): 140,
            ("Sevilla", "Huelva"): 90,
            ("Huelva", "Badajoz"): 130,
            ("Badajoz", "Caceres"): 80,
            ("Caceres", "Leon"): 220,
            ("Caceres", "Salamanca"): 70,
            ("Salamanca", "Leon"): 90,
            ("Madrid", "Ciudad Real"): 260,
            ("Toledo", "Cordoba"): 350,
            ("Sevilla", "Badajoz"): 260,
        },
        "start": "Madrid",
        "goal": "Leon",
    },


    # ROADMAP 5 — goal Sevilla; the optimal route goes through
    # Toledo and Cordoba
    5: {
        "roads": {
            ("Madrid", "Toledo"): 100,
            ("Madrid", "Ciudad Real"): 260,
            ("Toledo", "Cordoba"): 350,
            ("Toledo", "Badajoz"): 300,
            ("Cordoba", "Sevilla"): 140,
            ("Ciudad Real", "Caceres"): 290,
            ("Ciudad Real", "Leon"): 420,
        },
        "start": "Madrid",
        "goal": "Sevilla",
    },

    # ROADMAP 6 — ROADMAP 5's roads plus a direct Toledo-Leon road
    # (380), with goal Leon instead of Sevilla
    6: {
        "roads": {
            ("Madrid", "Ciudad Real"): 260,
            ("Madrid", "Toledo"): 100,
            ("Ciudad Real", "Caceres"): 290,
            ("Ciudad Real", "Leon"): 420,
            ("Toledo", "Leon"): 380,
            ("Toledo", "Badajoz"): 300,
            ("Toledo", "Cordoba"): 350,
            ("Cordoba", "Sevilla"): 140,
        },
        "start": "Madrid",
        "goal": "Leon",
    },

    # ROADMAP 7 — same roads, start and goal as ROADMAP 5
    7: {
        "roads": {
            ("Madrid", "Toledo"): 100,
            ("Madrid", "Ciudad Real"): 260,
            ("Toledo", "Badajoz"): 300,
            ("Toledo", "Cordoba"): 350,
            ("Cordoba", "Sevilla"): 140,
            ("Ciudad Real", "Caceres"): 290,
            ("Ciudad Real", "Leon"): 420,
        },
        "start": "Madrid",
        "goal": "Sevilla",
    },

    # ROADMAP 8 — same roads and goal as ROADMAP 6, listed in a
    # different order
    8: {
        "roads": {
            ("Madrid", "Toledo"): 100,
            ("Madrid", "Ciudad Real"): 260,
            ("Ciudad Real", "Caceres"): 290,
            ("Ciudad Real", "Leon"): 420,
            ("Toledo", "Leon"): 380,
            ("Toledo", "Badajoz"): 300,
            ("Toledo", "Cordoba"): 350,
            ("Cordoba", "Sevilla"): 140,
        },
        "start": "Madrid",
        "goal": "Leon",
    },

    # ROADMAP 9 — goal Coruna
    9: {
        "roads": {
            ("Madrid", "Barcelona"): 620,
            ("Madrid", "Pais Vasco"): 400,
            ("Madrid", "Leon"): 340,
            ("Pais Vasco", "Barcelona"): 570,
            ("Pais Vasco", "Leon"): 400,
            ("Leon", "Coruna"): 330,
        },
        "start": "Madrid",
        "goal": "Coruna",
    },

    # ROADMAP 10 — goal Lisboa
    10: {
        "roads": {
            ("Madrid", "Toledo"): 70,
            ("Madrid", "Avila"): 110,
            ("Toledo", "Caceres"): 260,
            ("Toledo", "Ciudad Real"): 120,
            ("Avila", "Salamanca"): 100,
            ("Caceres", "Badajoz"): 90,
            ("Ciudad Real", "Merida"): 270,
            ("Salamanca", "Merida"): 290,
            ("Badajoz", "Lisboa"): 230,
            ("Merida", "Lisboa"): 330,
        },
        "start": "Madrid",
        "goal": "Lisboa",
    }


}

if PROBLEM == "maze":
    if MAZE_TYPE not in MAZES:
        raise ValueError(f"MAZE_TYPE must be one of {sorted(MAZES)}, got {MAZE_TYPE!r}")

    MAZE_ROWS = 8
    MAZE_COLS = 10
    START = (1, 1)
    GOAL = (8, 10)
    MOVES = {"Up": (-1, 0), "Down": (1, 0), "Left": (0, -1), "Right": (0, 1)}
    WALLS = MAZES[MAZE_TYPE]

    NEIGHBOURS = {}
    for row in range(1, MAZE_ROWS + 1):
        for col in range(1, MAZE_COLS + 1):
            cell = (row, col)
            if cell in WALLS:
                continue
            edges = []
            for name, (dr, dc) in MOVES.items():
                next_cell = (row + dr, col + dc)
                in_bounds = 1 <= next_cell[0] <= MAZE_ROWS and 1 <= next_cell[1] <= MAZE_COLS
                if in_bounds and next_cell not in WALLS:
                    edges.append((name, next_cell, 1))
            NEIGHBOURS[cell] = edges

    print(f"Loaded MAZE {MAZE_TYPE} ({len(WALLS)} walls), start={START}, goal={GOAL}")

elif PROBLEM == "roadmap":
    if ROADMAP_TYPE not in ROADMAPS:
        raise ValueError(f"ROADMAP_TYPE must be one of {sorted(ROADMAPS)}, got {ROADMAP_TYPE!r}")

    chosen = ROADMAPS[ROADMAP_TYPE]
    ROADS = chosen["roads"]
    START = chosen["start"]
    GOAL = chosen["goal"]

    GRAPH = {}
    for (a, b), d in ROADS.items():
        GRAPH.setdefault(a, []).append((b, d))
        GRAPH.setdefault(b, []).append((a, d))

    NEIGHBOURS = {
        city: [(dest, dest, dist) for dest, dist in edges]
        for city, edges in GRAPH.items()
    }

    print(f"Loaded ROADMAP {ROADMAP_TYPE} ({len(ROADS)} roads), start={START}, goal={GOAL}")

else:
    raise ValueError(f"PROBLEM must be 'maze' or 'roadmap', got {PROBLEM!r}")


Loaded ROADMAP 5 (7 roads), start=Madrid, goal=Sevilla


## 4. The problem class

Both problems above reduce to the same question: *from this state, which
actions lead to which state, and at what cost?* `NEIGHBOURS` already
answers that for every state, no matter whether it holds `(row, col)`
maze cells or city names — so a single `Problem` subclass is enough for
both.


In [13]:
class GraphProblem(Problem):
    """A generic Problem over NEIGHBOURS: state -> [(action, next_state, cost), ...].

    Works for both the maze (states are (row, col) cells, actions are
    "Up"/"Down"/"Left"/"Right") and the road map (states are city names,
    and the action IS the destination city name), because both were
    already reduced to the same NEIGHBOURS shape in the problem
    definitions cell (PROBLEM / MAZE_TYPE / ROADMAP_TYPE).
    """

    def __init__(self, neighbour_order=None):
        """neighbour_order: optional dict {state: [preferred order of
        actions/destinations]}, used to experiment with how DFS depends on
        the order in which neighbours are tried (as in lab2)."""
        super().__init__(START)
        self.neighbour_order = neighbour_order or {}

    def is_goal(self, state):
        return state == GOAL

    def actions(self, state):
        edges = list(NEIGHBOURS.get(state, []))
        if state in self.neighbour_order:
            order = self.neighbour_order[state]
            edges.sort(key=lambda edge: order.index(edge[0]) if edge[0] in order else len(order))
        return [action for action, _next_state, _cost in edges]

    def result(self, state, action):
        for a, next_state, _cost in NEIGHBOURS[state]:
            if a == action:
                return next_state
        raise ValueError(f"No action {action!r} from state {state!r}")

    def step_cost(self, state, action, next_state):
        for a, ns, cost in NEIGHBOURS[state]:
            if a == action and ns == next_state:
                return cost
        raise ValueError(f"No edge {state} --{action}--> {next_state}")


## 5. Heuristics

To turn UCS into an **informed** search we need a heuristic `h(state)`
that estimates the cost from `state` to `GOAL` without ever
overestimating it (an *admissible* heuristic).

For `PROBLEM == "maze"`, states are `(row, col)` grid cells with
4-connected moves, every step costing 1 — three classic choices are:

- **Manhattan distance (L1)**: `h(s) = |Δrow| + |Δcol|`. The exact number
  of horizontal + vertical hops needed if there were no walls.
- **Chebyshev distance (L∞)**: `h(s) = max(|Δrow|, |Δcol|)`. On an empty
  grid with no walls, this would be the *exact* hop count if diagonal
  moves at cost 1 were allowed; here, with walls and only orthogonal
  moves, it is just a (looser) admissible lower bound that underestimates
  whenever both a row move and a column move are needed.
- **Euclidean distance (L2)**: `h(s) = sqrt(Δrow² + Δcol²)`, the straight
  "as the crow flies" distance.

All three are **admissible** (`h(s) <= h*(s)` for every state, since none
of them can exceed the number of orthogonal hops actually required — and
walls can only make the real distance longer, never shorter) and
**consistent** (`h(s) <= step_cost(s, a, s') + h(s')` for every move,
since a single step changes row/col by at most 1, so it changes each
estimate by at most 1). For every state `s` on this grid:

`h_chebyshev(s) <= h_euclidean(s) <= h_manhattan(s) <= h*(s)`

so **Manhattan is the most informed** heuristic here (the tightest lower
bound on the true cost); all three, being strictly positive (`h(s) > 0`)
for every non-goal state, can prune nodes that plain UCS still has to
expand, depending on the map and on ties in `f(n) = g(n) + h(n)`.
**Chebyshev is the least informed**
on this 4-connected grid — on an empty grid with no walls and diagonal
moves at cost 1 it would equal the true cost `h*`, but here (with walls,
no diagonals) it is only a looser admissible lower bound.

For `PROBLEM == "roadmap"`, states are city names — they carry no
`(row, col)` coordinates, so none of the three grid formulas above apply.
Instead we use a **manually tabulated straight-line ("as the crow flies")
distance to `GOAL`**, one entry per city: the same idea as the `h_SLD`
table for the Romania road map used to introduce Greedy best-first search
and A\* in Russell & Norvig [RN] — a human (or a map) supplies the
straight-line distance from every city to the goal city, and the search
treats it as `h(state)`.

For such a table to be a valid heuristic it must still be **admissible**
(`h(s) <= h*(s)`: a straight line is never longer than the road, possibly
winding, that connects the same two points) and **consistent**
(`h(s) <= step_cost(s, a, s') + h(s')` on every road `s -> s'`, the
triangle inequality applied edge by edge). Both `GOAL` and every road
distance change with `ROADMAP_TYPE`, so one single table would not stay
admissible across all ten maps: `SLD_TABLES` below therefore keeps
**one table per `ROADMAP_TYPE`**, each checked against that map's own
road costs.

These tables are **illustrative**: the numbers are hand-adjusted to each
map's own made-up road distances in kilometres, not real geographic
straight-line distances. What matters for A\* and Greedy to behave
correctly is not geographic accuracy but that each table stays
admissible and consistent with the road costs of the map it belongs to.

If a `ROADMAP_TYPE` has no entry in `SLD_TABLES`, `HEURISTICS` is left
empty and this notebook skips informed search (Greedy and A\*) for that
map, while BFS, DFS and UCS keep working as usual.

In [14]:

# Straight-line-distance tables, one per ROADMAP_TYPE — the same idea as
# the h_SLD table for Russell & Norvig's Romania map [RN]. These are
# illustrative straight-line distances (not computed from real
# coordinates), hand-adjusted so that, against THIS map's made-up road
# costs, they stay admissible (h <= h*, the true shortest-road-distance
# to GOAL) and consistent (h(u) <= cost(u, v) + h(v) on every road).
SLD_TABLES = {
    # ROADMAP 1 — goal Valencia. True shortest road distance (h*) on this
    # map: Albacete 180, Cuenca 250, Madrid 400, Valencia 0.
    1: {"Madrid": 320, "Cuenca": 175, "Albacete": 160, "Valencia": 0},
    # ROADMAP 2 — same cities and roads as ROADMAP 1 (listed in a
    # different order), so the same table stays admissible/consistent.
    2: {"Madrid": 320, "Cuenca": 175, "Albacete": 160, "Valencia": 0},
    # ROADMAP 3 — goal Leon. True shortest road distance (h*) on this
    # map: Salamanca 90, Caceres 160, Badajoz 240, Huelva 370,
    # Sevilla 460, Cordoba 600, Ciudad Real 780, Toledo 900, Madrid 1000,
    # Leon 0.
    3: {
        "Madrid": 800, "Toledo": 720, "Ciudad Real": 620, "Cordoba": 480,
        "Sevilla": 370, "Huelva": 300, "Badajoz": 190, "Caceres": 130,
        "Salamanca": 70, "Leon": 0,
    },
    # ROADMAP 4 — same cities and roads as ROADMAP 3 (listed in a
    # different order), so the same table stays admissible/consistent.
    4: {
        "Madrid": 800, "Toledo": 720, "Ciudad Real": 620, "Cordoba": 480,
        "Sevilla": 370, "Huelva": 300, "Badajoz": 190, "Caceres": 130,
        "Salamanca": 70, "Leon": 0,
    },
    # ROADMAP 5 — goal Sevilla. True shortest road distance (h*) on this
    # map: Cordoba 140, Toledo 490, Madrid 590, Badajoz 790, Ciudad Real 850,
    # Caceres 1140, Leon 1270, Sevilla 0.
    5: {
        "Madrid": 500, "Toledo": 420, "Ciudad Real": 740, "Cordoba": 120,
        "Sevilla": 0, "Badajoz": 700, "Caceres": 1000, "Leon": 1100,
    },
    # ROADMAP 6 — goal Leon. True shortest road distance (h*) on this
    # map: Toledo 380, Ciudad Real 420, Madrid 480, Badajoz 680,
    # Caceres 710, Cordoba 730, Sevilla 870, Leon 0.
    6: {
        "Madrid": 400, "Toledo": 320, "Ciudad Real": 340, "Cordoba": 620,
        "Sevilla": 740, "Badajoz": 580, "Caceres": 600, "Leon": 0,
    },

    # ROADMAP 7 — same cities, roads and goal as ROADMAP 5, so the
    # same table stays admissible/consistent.
    7: {
        "Madrid": 500, "Toledo": 420, "Ciudad Real": 740, "Cordoba": 120,
        "Sevilla": 0, "Badajoz": 700, "Caceres": 1000, "Leon": 1100,
    },

    8: {
        "Madrid": 400, "Toledo": 320, "Ciudad Real": 340, "Cordoba": 620,
        "Sevilla": 740, "Badajoz": 580, "Caceres": 600, "Leon": 0,
    },

    9: {
    "Madrid": 510, "Barcelona": 890, "Pais Vasco": 520, "Leon": 270, "Coruna": 0,
    },

    10: {
        "Madrid": 500, "Toledo": 440, "Avila": 480,
        "Caceres": 260, "Ciudad Real": 400, "Salamanca": 380,
        "Badajoz": 200, "Merida": 280, "Lisboa": 0,
    },

    
}


In [15]:

def manhattan(a, b):
    """|Δrow| + |Δcol| (L1). Exact hop count with no walls and no
    diagonal moves — the most informed of the three on this grid."""
    (r1, c1), (r2, c2) = a, b
    return abs(r1 - r2) + abs(c1 - c2)


def chebyshev(a, b):
    """max(|Δrow|, |Δcol|) (L∞). On an empty grid with no walls, this
    would be the exact hop count if diagonal moves at cost 1 were
    allowed; here, with walls and only orthogonal moves, it is just an
    admissible lower bound that underestimates."""
    (r1, c1), (r2, c2) = a, b
    return max(abs(r1 - r2), abs(c1 - c2))


def euclidean(a, b):
    """sqrt(Δrow² + Δcol²) (L2). Straight-line distance; always
    <= Manhattan, so admissible but less informed."""
    (r1, c1), (r2, c2) = a, b
    return math.hypot(r1 - r2, c1 - c2)



def straight_line(state, goal):
    """h_SLD(state): manually tabulated straight-line distance to `goal`,
    read from the SLD_TABLES entry for the active ROADMAP_TYPE. `goal`
    must be the active GOAL — the table only knows distances to it."""
    if ROADMAP_TYPE not in SLD_TABLES:
        raise ValueError(
            f"No straight-line table in SLD_TABLES for "
            f"ROADMAP_TYPE={ROADMAP_TYPE}: straight_line cannot be used "
            f"for this map."
        )
    if goal != ROADMAPS[ROADMAP_TYPE]["goal"]:
        raise ValueError(
            f"straight_line only has distances to "
            f"{ROADMAPS[ROADMAP_TYPE]['goal']!r} (the GOAL of "
            f"ROADMAP_TYPE={ROADMAP_TYPE}), not {goal!r}"
        )
    return SLD_TABLES[ROADMAP_TYPE][state]


if PROBLEM == "maze":
    HEURISTICS = {"manhattan": manhattan, "chebyshev": chebyshev, "euclidean": euclidean}
elif PROBLEM == "roadmap":
    # No SLD entry for this ROADMAP_TYPE -> skip informed search (Greedy, A*)
    # for this map; HEURISTICS stays empty and BFS/DFS/UCS are unaffected.
    HEURISTICS = {"straight_line": straight_line} if ROADMAP_TYPE in SLD_TABLES else {}

print("Heuristic values from START to GOAL:")
if not HEURISTICS:
    print(
        f"No straight-line table in SLD_TABLES for ROADMAP_TYPE={ROADMAP_TYPE}: "
        "informed search (Greedy, A*) is skipped for this map."
    )
else:
    for name, h in HEURISTICS.items():
        print(f"  {name:14} h(START, GOAL) = {h(START, GOAL):.3f}")


Heuristic values from START to GOAL:
  straight_line  h(START, GOAL) = 500.000


## 6. Visualize the solution

Two very different ways to *see* a path, one per problem:

- **maze**: a text grid, `S`/`G` for start/goal, `#` for walls, `o` for the
  cells on the path.
- **roadmap**: the sequence of cities with the distance (in km) of each
  road, from `START` to `GOAL`.

`show_solution(path=None, problem=None)` is the single entry point used
from here on: it looks at `PROBLEM` and dispatches to `show_maze` or
`show_route`. Calling it with no path at all (as below) simply shows the
problem itself, before any search has run.


In [16]:
def show_maze(path=None):
    path_cells = set(path or [])
    print("   " + "".join(f"{c:>2}" for c in range(1, MAZE_COLS + 1)))
    for r in range(1, MAZE_ROWS + 1):
        row_chars = []
        for c in range(1, MAZE_COLS + 1):
            cell = (r, c)
            if cell == START:
                ch = "S"
            elif cell == GOAL:
                ch = "G"
            elif cell in WALLS:
                ch = "#"
            elif cell in path_cells:
                ch = "o"
            else:
                ch = "."
            row_chars.append(f"{ch:>2}")
        print(f"{r:>2} " + "".join(row_chars))


In [17]:
def show_route(path, problem=None):
    if not path:
        print("(no route found)")
        return
    problem = problem or GraphProblem()
    total = 0
    parts = [path[0]]
    for a, b in zip(path, path[1:]):
        dist = problem.step_cost(a, b, b)
        total += dist
        parts.append(f"--{dist}km-->")
        parts.append(b)
    print(" ".join(parts))
    print(f"Total: {total} km over {len(path) - 1} step(s)")


In [18]:
def show_solution(path=None, problem=None):
    """Dispatch to show_maze or show_route depending on the active PROBLEM."""
    if PROBLEM == "maze":
        show_maze(path)
    elif PROBLEM == "roadmap":
        show_route(path, problem)
    else:
        raise ValueError(f"Unknown PROBLEM: {PROBLEM!r}")


print(f"Problem (no route yet) — PROBLEM={PROBLEM!r}, start={START}, goal={GOAL}:")
show_solution()


Problem (no route yet) — PROBLEM='roadmap', start=Madrid, goal=Sevilla:
(no route found)


## 7. Step-by-step execution

Now we care not only about the final route, but about **how each
algorithm got there**. Each cell below runs one strategy with
`search_tree()` (defined in section 3) and prints the **search tree** it
builds:

- the `[#k]` number is the **order of expansion**: notice how BFS expands
  level by level, DFS goes down one branch to the bottom, and UCS always
  expands the node with the lowest accumulated cost `g`;
- `(in frontier)` are nodes generated but never expanded;
- `*` marks the nodes on the solution path.

Compare the three trees: same problem, same generic function, only how
the frontier is managed changes.

On the maze these trees can get long and wide (many cells, small step
cost), so they are easiest to read on a small `MAZE_TYPE`. The road map,
with far fewer states, keeps the tree short enough to also see effects
such as a node being generated more than once.


In [19]:
# BFS — the frontier is a FIFO queue: expands level by level
problem = GraphProblem()
_ = search_tree(problem, "bfs")


Search tree — BFS  (Madrid -> goal)
*Madrid (g=0) [#1]
├── --100--> *Toledo (g=100) [#2]
│   ├── --100--> Madrid (pruned: already visited)
│   ├── --350--> *Cordoba (g=450) [#4]
│   │   ├── --350--> Toledo (pruned: already visited)
│   │   └── --140--> *Sevilla (g=590) GOAL
│   └── --300-->  Badajoz (g=400) [#5]
│       └── --300--> Toledo (pruned: already visited)
└── --260-->  Ciudad Real (g=260) [#3]
    ├── --260--> Madrid (pruned: already visited)
    ├── --290-->  Caceres (g=550) [#6]
    │   └── --290--> Ciudad Real (pruned: already visited)
    └── --420-->  Leon (g=680) [#7]
        └── --420--> Ciudad Real (pruned: already visited)

Solution: Madrid -> Toledo -> Cordoba -> Sevilla  (3 step(s), cost 590, 7 node(s) expanded)


In [20]:
# DFS — the frontier is a LIFO stack: goes down one branch to the bottom
problem = GraphProblem()
_ = search_tree(problem, "dfs")


Search tree — DFS  (Madrid -> goal)
*Madrid (g=0) [#1]
├── --100--> *Toledo (g=100) [#2]
│   ├── --100--> Madrid (pruned: already visited)
│   ├── --350--> *Cordoba (g=450) [#3]
│   │   ├── --350--> Toledo (pruned: already visited)
│   │   └── --140--> *Sevilla (g=590) GOAL
│   └── --300-->  Badajoz (g=400) (in frontier)
└── --260-->  Ciudad Real (g=260) (in frontier)

Solution: Madrid -> Toledo -> Cordoba -> Sevilla  (3 step(s), cost 590, 3 node(s) expanded)


In [21]:
# UCS — the frontier is a priority queue ordered by accumulated cost g
problem = GraphProblem()
_ = search_tree(problem, "ucs")


Search tree — UCS  (Madrid -> goal)
*Madrid (g=0) [#1]
├── --100--> *Toledo (g=100) [#2]
│   ├── --100--> Madrid (pruned: already visited)
│   ├── --350--> *Cordoba (g=450) [#5]
│   │   ├── --350--> Toledo (pruned: already visited)
│   │   └── --140--> *Sevilla (g=590) GOAL
│   └── --300-->  Badajoz (g=400) [#4]
│       └── --300--> Toledo (pruned: already visited)
└── --260-->  Ciudad Real (g=260) [#3]
    ├── --260--> Madrid (pruned: already visited)
    ├── --290-->  Caceres (g=550) [#6]
    │   └── --290--> Ciudad Real (pruned: already visited)
    └── --420-->  Leon (g=680) (in frontier)

Solution: Madrid -> Toledo -> Cordoba -> Sevilla  (3 step(s), cost 590, 6 node(s) expanded)


## 8. Run BFS, DFS and UCS

Before comparing them automatically, let's run each strategy separately
to calmly see what we pass to `search()` and what we get back. Yes, the
code in the next three cells is nearly identical — in practice you would
use a loop, like `compare_heuristics()` does in the last section — but
this way you see, cell by cell, what each call does.


In [22]:
problem = GraphProblem()


In [23]:
# BFS
bfs_result = search(problem, "bfs")
print("BFS:")
show_solution(bfs_result["path"], problem)


BFS:
Madrid --100km--> Toledo --350km--> Cordoba --140km--> Sevilla
Total: 590 km over 3 step(s)


In [24]:
# DFS
dfs_result = search(problem, "dfs")
print("DFS:")
show_solution(dfs_result["path"], problem)


DFS:
Madrid --100km--> Toledo --350km--> Cordoba --140km--> Sevilla
Total: 590 km over 3 step(s)


In [25]:
# UCS
ucs_result = search(problem, "ucs")
print("UCS:")
show_solution(ucs_result["path"], problem)


UCS:
Madrid --100km--> Toledo --350km--> Cordoba --140km--> Sevilla
Total: 590 km over 3 step(s)


## 9. DFS depends on the order of the neighbours

`GraphProblem` accepts an optional `neighbour_order` so you can force DFS
to try a specific neighbour of a state first, without changing anything
else. Here we simply take the neighbours of `START` in two orders — the
order they already appear in `NEIGHBOURS` ("natural") and the reverse
("inverted") — and watch how DFS's route changes.


In [26]:
natural_order = [action for action, _next_state, _cost in NEIGHBOURS[START]]
inverted_order = list(reversed(natural_order))

print("Natural order of neighbours from START:", natural_order)
problem_natural = GraphProblem(neighbour_order={START: natural_order})
r_natural = search(problem_natural, "dfs")
show_solution(r_natural["path"], problem_natural)

print("\nInverted order of neighbours from START:", inverted_order)
problem_inverted = GraphProblem(neighbour_order={START: inverted_order})
r_inverted = search(problem_inverted, "dfs")
show_solution(r_inverted["path"], problem_inverted)


Natural order of neighbours from START: ['Toledo', 'Ciudad Real']
Madrid --100km--> Toledo --350km--> Cordoba --140km--> Sevilla
Total: 590 km over 3 step(s)

Inverted order of neighbours from START: ['Ciudad Real', 'Toledo']
Madrid --100km--> Toledo --350km--> Cordoba --140km--> Sevilla
Total: 590 km over 3 step(s)


In [27]:
def ucs_trace(problem):
    """A traced version of UCS, just for teaching: same rules as search(),
    but it prints what happens to the frontier at every expansion."""
    root = Node(problem.initial_state)
    counter = itertools.count()
    frontier = []
    heapq.heappush(frontier, (root.path_cost, next(counter), root))
    best_shown = {root.state: root.path_cost}  # best cost seen so far per state

    step = 0
    while frontier:
        g, _, node = heapq.heappop(frontier)
        if node.state in best_shown and g > best_shown[node.state]:
            continue  # a stale, already-replaced frontier entry

        step += 1
        print(f"{step}. Expand {node.state} (g={g})")

        if problem.is_goal(node.state):
            print(f"   -> goal selected with g={g}: stop")
            return node

        for action in problem.actions(node.state):
            next_state = problem.result(node.state, action)
            new_g = node.path_cost + problem.step_cost(node.state, action, next_state)
            if next_state not in best_shown or new_g < best_shown[next_state]:
                note = "in frontier, replaced" if next_state in best_shown else "added to frontier"
                print(f"   {next_state}: g={new_g} ({note})")
                best_shown[next_state] = new_g
                child = Node(next_state, node, action, new_g, node.depth + 1)
                heapq.heappush(frontier, (new_g, next(counter), child))
    return None


print(f"UCS trace, {START} -> {GOAL}:")
problem = GraphProblem()
_ = ucs_trace(problem)


UCS trace, Madrid -> Sevilla:
1. Expand Madrid (g=0)
   Toledo: g=100 (added to frontier)
   Ciudad Real: g=260 (added to frontier)
2. Expand Toledo (g=100)
   Cordoba: g=450 (added to frontier)
   Badajoz: g=400 (added to frontier)
3. Expand Ciudad Real (g=260)
   Caceres: g=550 (added to frontier)
   Leon: g=680 (added to frontier)
4. Expand Badajoz (g=400)
5. Expand Cordoba (g=450)
   Sevilla: g=590 (added to frontier)
6. Expand Caceres (g=550)
7. Expand Sevilla (g=590)
   -> goal selected with g=590: stop


## 10. Compare informed search with different heuristics

`compare_heuristics` runs BFS, DFS and UCS as references, plus **A\*** and
**Greedy best-first**, each with every heuristic currently in
`HEURISTICS`, and prints the same kind of metrics table for all of them;
if `HEURISTICS` is empty (no SLD table for this map), only the
BFS/DFS/UCS rows are shown.

DFS shows up as three separate rows, one per repeated-state policy:
`DFS nc` (`repeated="No_control"`), `DFS pchk`
(`"Remember_path_checking"`) and `DFS all` (`"Remember_all_visited"`).
`DFS nc` usually hits the `max_expansions` limit (`found=False`, 10000
nodes expanded), because without any control of repeated states it keeps
going back and forth along the same road or cell.


In [28]:
def compare_heuristics(problem, max_expansions=10_000):
    rows = [("BFS", None, search(problem, "bfs", max_expansions=max_expansions))]
    rows.append(("DFS nc", None, search(problem, "dfs", repeated="No_control", max_expansions=max_expansions)))
    rows.append(("DFS pchk", None, search(problem, "dfs", repeated="Remember_path_checking", max_expansions=max_expansions)))
    rows.append(("DFS all", None, search(problem, "dfs", repeated="Remember_all_visited",max_expansions=max_expansions)))
    rows.append(("UCS", None, search(problem, "ucs", max_expansions=max_expansions)))
    for strategy in ("greedy","astar"):
        for name in HEURISTICS:
            r = search(problem, strategy, heuristic=name, max_expansions=max_expansions)
            rows.append((strategy.upper(), name, r))

    header = (f"{'strategy':8} | {'heuristic':14} | {'found':5} | {'steps':>5} | "
              f"{'cost':>6} | {'expanded':>8} | {'max frontier':>12} | "
              f"{'max memory':>10} | {'time (ms)':>9}")
    print(header)
    print("-" * len(header))
    for strategy_name, heuristic_name, r in rows:
        steps = r["steps"] if r["found"] else "-"
        cost = r["path_cost"] if r["found"] else "-"
        print(f"{strategy_name:8} | {heuristic_name or '-':14} | {str(r['found']):5} | "
              f"{steps!s:>5} | {cost!s:>6} | {r['nodes_expanded']:>8} | "
              f"{r['max_frontier_size']:>12} | {r['max_memory']:>10} | {r['time_ms']:>9.3f}")
    if not HEURISTICS:
        print("\nNo GREEDY/ASTAR rows: HEURISTICS is empty for this map "
              "(no SLD table for this ROADMAP_TYPE).")
    return rows

print("\nProblem:", PROBLEM)

if PROBLEM == "maze":
    print("Type:", MAZE_TYPE)
else:
    print("Type:", ROADMAP_TYPE)

print("\n")

problem = GraphProblem()
heuristic_comparison = compare_heuristics(problem)



Problem: roadmap
Type: 5


strategy | heuristic      | found | steps |   cost | expanded | max frontier | max memory | time (ms)
-----------------------------------------------------------------------------------------------------
BFS      | -              | True  |     3 |    590 |        7 |            4 |          8 |     0.022
DFS nc   | -              | False |     - |      - |    10000 |        15001 |      25001 |    17.211
DFS pchk | -              | True  |     3 |    590 |        3 |            3 |          6 |     0.019
DFS all  | -              | True  |     3 |    590 |        3 |            3 |          6 |     0.010
UCS      | -              | True  |     3 |    590 |        6 |            4 |          8 |     0.014
GREEDY   | straight_line  | True  |     3 |    590 |        3 |            3 |          6 |     0.011
ASTAR    | straight_line  | True  |     3 |    590 |        3 |            3 |          6 |     0.009


Pick a heuristic below and re-run: A\* stays **optimal** no matter which
entry of `HEURISTICS` you choose, because all of them are admissible —
only the number of expanded nodes changes. **Greedy** best-first, on the
other hand, is *not* guaranteed to be optimal with any of them, since it
ignores `g(n)` entirely and can commit to a route that only *looks* close
to the goal. If `HEURISTICS` is empty for this map, the cell below is
skipped instead.


In [29]:
if not HEURISTICS:
    print("No heuristic available for this map (HEURISTICS is empty): "
          "skipping A* here.")
else:
    HEURISTIC = next(iter(HEURISTICS))  # change to any other key of HEURISTICS,
                                         # e.g. list(HEURISTICS)[1], to compare
                                         # (only the maze has more than one;
                                         # the roadmap only has straight_line)
    astar_result = search(problem, "astar", heuristic=HEURISTIC)
    print(f"A* with {HEURISTIC} heuristic: found={astar_result['found']}, "
          f"steps={astar_result['steps']}, cost={astar_result['path_cost']}, "
          f"nodes_expanded={astar_result['nodes_expanded']}")
    show_solution(astar_result["path"], problem)


A* with straight_line heuristic: found=True, steps=3, cost=590, nodes_expanded=3
Madrid --100km--> Toledo --350km--> Cordoba --140km--> Sevilla
Total: 590 km over 3 step(s)
